## Proceso de limpieza en tabla de empleados

#### Importacion de librerias y analisis general

In [21]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import unicodedata as nc

In [255]:
df_empleados2 = pd.read_excel('./datos_crudos/empleados_empresa_dirty.xlsx')

In [256]:
df_empleados2

,id_empleado,nombre,edad,sexo,departamento,salario,fecha_ingreso
0,101,Lucia Fernández,30,M,Recursos humanos,180.000,03/04/2019
1,102,Lucia Fernández,30,NaN,Recursos humanos,NaN,NaN
2,103,Juan Perez,25,M,ventas,130000,2020-10-31
3,104,Pedro martinez,NaN,NaN,Recursos humanos,dos cientos mil,09/05/2020
4,105,Ana Gómez,150,M,NaN,130000,2018-13-01
...,...,...,...,...,...,...,...
145,246,Ana Gómez,veinticinco,m,Recursos humanos,130000,24/02/2020
146,247,NaN,28,M,IT,dos cientos mil,NaN
147,248,Ana Gómez,45,f,Ventas,130000,2018-13-01
148,249,NaN,30,f,Ventas,NaN,01/02/2018


In [257]:
df_empleados2.shape

(150, 7)

In [258]:
df_empleados2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   id_empleado    150 non-null    int64 
 1   nombre         127 non-null    object
 2   edad           129 non-null    object
 3   sexo           121 non-null    object
 4   departamento   135 non-null    object
 5   salario        123 non-null    object
 6   fecha_ingreso  112 non-null    object
dtypes: int64(1), object(6)
memory usage: 8.3+ KB


In [259]:
df_empleados2.apply(lambda col: col.duplicated().sum())

id_empleado        0
nombre           143
edad             142
sexo             144
departamento     143
salario          144
fecha_ingreso     69
dtype: int64

In [260]:
df_empleados2.isna().sum()

id_empleado       0
nombre           23
edad             21
sexo             29
departamento     15
salario          27
fecha_ingreso    38
dtype: int64

#### Conclusión :

El dataset está compuesto por 150 filas y 7 columnas. En una primera inspección se detecta la presencia de valores faltantes en todas las columnas excepto id_empleado, lo cual es esperable dado que esta última funciona como identificador único. La cantidad de valores nulos es relativamente similar en la mayoría de las variables (entre 20 y 30 registros), aunque la columna fecha_ingreso presenta una cantidad levemente superior.

Se observan múltiples valores duplicados en distintas columnas. En variables categóricas como sexo y departamento, la repetición es esperable debido a la naturaleza de los datos. Lo mismo ocurre en variables cuantitativas como edad y salario, y en la variable temporal fecha_ingreso. Sin embargo, la presencia de posibles duplicados en la columna nombre, combinados con inconsistencias en otros campos, requiere un análisis más profundo para determinar si se trata de registros repetidos o inconsistencias de carga.

En cuanto a los tipos de datos, se identifican inconsistencias importantes: las columnas edad, salario y fecha_ingreso se encuentran almacenadas como tipo object (string), cuando deberían ser tratadas respectivamente como variables numéricas y temporales. Esto indica la necesidad de un proceso de conversión y estandarización. Asimismo, se detectan errores de formato, como variaciones entre mayúsculas y minúsculas, diferencias en la escritura de categorías (por ejemplo, distintas formas de representar el sexo o el departamento), y valores numéricos almacenados como texto, incluyendo algunos escritos en palabras.

Adicionalmente, en la columna edad se observan valores atípicos o anómalos (por ejemplo, edades de 150 años), que resultan poco plausibles en el contexto del dataset. Estos registros deberán analizarse para determinar si se trata de errores de carga, valores mal ingresados o casos que deban ser depurados mediante reglas de validación.

Finalmente, se identifican filas con alto grado de similitud entre sí, algunas con información complementaria y otras con valores contradictorios. Será necesario aplicar criterios de limpieza y reglas de negocio para determinar qué registros conservar, priorizando aquellos con mayor completitud o coherencia interna.

#### Estandarización de formatos

In [262]:
col_str = df_empleados2.select_dtypes(include = 'object').columns
df_empleados2[col_str] = df_empleados2[col_str].apply(
lambda col: col.astype(str)
.str.strip()
.str.lower()
.str.normalize('NFKD')
.str.encode('ascii',errors='ignore')
.str.decode('utf-8')
)

#### Tareas realizadas :
Se estandarizaron todas las columnas de tipo texto (object), convirtiendo los valores a string, eliminando espacios en blanco, unificando a minúsculas y removiendo tildes y caracteres especiales.
El objetivo es reducir inconsistencias y mejorar la calidad de los datos antes del análisis.

#### Tratamiento columna 'salario'

In [429]:
df_empleados2['salario'].unique()

array([180000., 130000.,     nan, 200000., 120000.])

In [265]:
df_empleados2['salario'] = df_empleados2['salario'].str.replace('.','',regex=False)

In [266]:
df_empleados2['salario'] = df_empleados2['salario'].str.replace('dos cientos mil','200000',regex=False)

In [444]:
df_empleados2 = df_empleados2[~df_empleados2['salario'].isnull()]

In [445]:
df_empleados2

,id_empleado,nombre,edad,sexo,departamento,salario,fecha_ingreso
0,101,lucia fernandez,30,m,recursos humanos,180000.0,2019-04-03
2,103,juan perez,25,m,ventas,130000.0,2020-10-31
6,107,juan perez,25,f,recursos humanos,180000.0,NaT
10,111,juan perez,25,f,ventas,200000.0,2019-07-01
11,112,pedro martinez,45,f,recursos humanos,130000.0,NaT
14,115,juan perez,40,f,recursos humanos,200000.0,2019-06-27
16,117,juan perez,45,f,it,120000.0,2022-03-28
23,124,ana gomez,25,m,it,180000.0,2018-04-29
24,125,pedro martinez,30,m,recursos humanos,200000.0,2018-05-22
25,126,juan perez,40,f,ventas,180000.0,NaT


In [438]:
df_empleados2['salario'] = df_empleados2['salario'].astype(float)

In [498]:
df_empleados_copy = df_empleados_copy[df_empleados_copy['id_empleado'] != 201]

#### Tareas realizadas en columna 'salario'
La variable se encontraba en formato object. En primer lugar, se procedió a la eliminación de los puntos presentes en los valores numéricos. Posteriormente, se reemplazaron los valores escritos en texto (por ejemplo, "doscientos mil") por su equivalente numérico. Finalmente, la variable fue convertida al tipo de dato float. Los valores nulos se mantuvieron con el objetivo de evaluar posteriormente si es posible su recuperación o imputación.  Finalmente, debido a inconsistencias detectadas en los registros, y siguiendo lineamientos proporcionados por la empresa, se decidió eliminar aquellas filas que presentaran valores nulos y/0 tambien datos irreales.

#### Tratamiento columna 'sexo'

In [271]:
df_empleados2['sexo'] = df_empleados2['sexo'].str.replace('femenino','f',regex=False)

In [284]:
df_empleados2['sexo'].unique()

array(['m', 'f'], dtype=object)

In [278]:
df_empleados2 = df_empleados2[df_empleados2['sexo'] != 'nan']

#### Tareas realizadas en columna 'sexo'
Se identificaron valores inconsistentes como "femenino", además de las categorías "m" y "f".  
Se procedió a normalizar la columna, unificando los registros y reemplazando "femenino" por "f", con el fin de estandarizar las categorías y facilitar el análisis posterior.  Finalmente, debido a inconsistencias detectadas en los registros, y siguiendo lineamientos proporcionados por la empresa, se decidió eliminar aquellas filas que presentaran valores nulos y/0 tambien datos irreales.

#### Tratamiento columna 'nombre'

In [449]:
df_empleados2['nombre'].unique()

array(['lucia fernandez', 'juan perez', 'pedro martinez', 'ana gomez',
       'maria lopez'], dtype=object)

In [454]:
df_empleados2 = df_empleados2[df_empleados2['nombre'] != 'nan']

#### Tareas realizadas en la columna 'nombre'

Debido a inconsistencias detectadas en los registros, y siguiendo lineamientos proporcionados por la empresa, se decidió eliminar aquellas filas que presentaran valores nulos.  
De esta manera, se trabajó únicamente con registros completos y consistentes para garantizar mayor confiabilidad en el análisis.

#### Tratamiento columna 'edad'

In [366]:
df_empleados2.loc[df_empleados2['edad'] == 'veinticinco','edad'] = 25

In [370]:
df_empleados2 = df_empleados2[df_empleados2['edad'] != 'nan']

In [379]:
df_empleados2 = df_empleados2[df_empleados2['edad'] != '150']

In [461]:
df_empleados2['edad'].unique()

array(['30', '25', 25, '45', '40', '28'], dtype=object)

#### Tareas realizadas en columna 'edad'
En la columna edad se realizó la conversión de valores numéricos que estaban escritos en texto (por ejemplo, "veinticinco") a formato numérico para unificar el tipo de dato. Además, se eliminó el registro que contenía una edad irreal (150), ya que representa un valor fuera de rango. Finalmente, debido a inconsistencias detectadas en los registros, y siguiendo lineamientos proporcionados por la empresa, se decidió eliminar aquellas filas que presentaran valores nulos.

#### Realizamos una copia del Dataframe

In [569]:
df_empleados_copy = df_empleados2.copy()

#### Tratamiento columna 'fecha_ingreso'

In [570]:
df_empleados_copy1['fecha_ingreso'] = pd.to_datetime(df_empleados_copy['fecha_ingreso'],format= 'mixed',dayfirst=True)

In [571]:
df_empleados_copy = df_empleados_copy[~df_empleados_copy['fecha_ingreso'].isnull()]

In [572]:
df_empleados_copy.info()

<class 'pandas.core.frame.DataFrame'>
Index: 40 entries, 0 to 147
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   id_empleado    40 non-null     int64         
 1   nombre         40 non-null     object        
 2   edad           40 non-null     object        
 3   sexo           40 non-null     object        
 4   departamento   40 non-null     object        
 5   salario        40 non-null     float64       
 6   fecha_ingreso  40 non-null     datetime64[ns]
dtypes: datetime64[ns](1), float64(1), int64(1), object(4)
memory usage: 2.5+ KB


#### Tareas realizadas en columna 'fecha_ingreso'
Se realiza la convercion de la variable fecha_ingreso a datatime formato fecha. Finalmente, se descartaron las filas con valores nulos debido a inconsistencias significativas en la tabla, con el objetivo de garantizar mayor calidad y coherencia en el análisis posterior.


#### Tratamiento columna 'departamento'

In [573]:
df_empleados_copy = df_empleados_copy[df_empleados_copy['departamento'] != 'nan']

#### Tareas en columna 'departamento'
Debido a inconsistencias detectadas en los registros, y siguiendo lineamientos proporcionados por la empresa, se decidió eliminar aquellas filas que presentaran valores nulos.

#### Depuración de inconsistencias lógicas en los datos

##### Empleada 'lucia fernandez'

In [578]:
df_empleados_copy[df_empleados_copy['nombre'] == 'lucia fernandez']

,id_empleado,nombre,edad,sexo,departamento,salario,fecha_ingreso
51,152,lucia fernandez,28,f,ventas,120000.0,2022-04-29
68,169,lucia fernandez,45,f,recursos humanos,180000.0,2020-04-11
113,214,lucia fernandez,45,f,ventas,120000.0,2019-08-22


In [575]:
df_empleados_copy = df_empleados_copy[df_empleados_copy['id_empleado'] != 201]

In [576]:
df_empleados_copy = df_empleados_copy[df_empleados_copy['id_empleado'] != 172] 

In [577]:
df_empleados_copy = df_empleados_copy[df_empleados_copy['id_empleado'] != 101]

##### conclusion :
Se detectó un error en el id_empleado, ya que existen registros con el mismo nombre, apellido y edad pero con distinto identificador. Debido a la coincidencia de los datos personales, se asume que corresponde a la misma persona, quien ingresó en 2019 en el área de Ventas y luego pasó a Recursos Humanos en 2020. Asimismo, se identifica otro empleado con el mismo nombre pero distinta edad, que ingresó en 2022 en el sector de Ventas.

##### empleado 'juan perez'

In [579]:
df_empleados_copy = df_empleados_copy[~(
(df_empleados_copy['nombre']=='juan perez') &
(df_empleados_copy['sexo'] == 'f'))
]

In [580]:
df_empleados_copy[df_empleados_copy['nombre'] == 'juan perez']

,id_empleado,nombre,edad,sexo,departamento,salario,fecha_ingreso
2,103,juan perez,25,m,ventas,130000.0,2020-10-31


##### Conclusión :
Se tiene un solo empleado llamado 'juan perez' con los datos correctos referidos al sector, salario y genero

##### empleada 'ana gomez'

In [581]:
df_empleados_copy[df_empleados_copy['nombre'] == 'ana gomez']

,id_empleado,nombre,edad,sexo,departamento,salario,fecha_ingreso
23,124,ana gomez,25,m,it,180000.0,2018-04-29
27,128,ana gomez,40,m,ventas,130000.0,2018-05-12
43,144,ana gomez,25,f,ventas,200000.0,2019-10-06
70,171,ana gomez,40,f,recursos humanos,130000.0,2020-08-27
75,176,ana gomez,28,m,ventas,200000.0,2021-11-09
84,185,ana gomez,25,f,it,200000.0,2018-01-13
124,225,ana gomez,30,f,ventas,200000.0,2019-06-29
126,227,ana gomez,30,f,recursos humanos,180000.0,2018-01-13
131,232,ana gomez,25,f,it,180000.0,2018-08-03
145,246,ana gomez,25,m,recursos humanos,130000.0,2020-02-24


In [582]:
df_empleados_copy = df_empleados_copy[
    ~(
        (df_empleados_copy['nombre'] == 'ana gomez') &
        (df_empleados_copy['sexo'] == 'm')
    )
]

In [583]:
df_empleados_copy = df_empleados_copy[
    ~(
        (df_empleados_copy['nombre'] == 'ana gomez') &
        (~df_empleados_copy['id_empleado'].isin([185, 227, 248]))
    )
]

##### Conclusión :
Se observa el ingreso de tres empleados con el mismo nombre, pero con distinto id_empleado, edad, departamento y salario en la misma fecha. Debido a estas diferencias estructurales, se concluye que se trata de tres personas distintas.

##### empleado 'maria lopez'

In [586]:
df_empleados_copy[df_empleados_copy['nombre'] == 'maria lopez']

,id_empleado,nombre,edad,sexo,departamento,salario,fecha_ingreso
32,133,maria lopez,28,f,it,200000.0,2018-04-21
130,231,maria lopez,45,f,it,200000.0,2022-07-26


In [585]:
df_empleados_copy = df_empleados_copy[
    ~(
        (df_empleados_copy['nombre'] == 'maria lopez') &
        (~df_empleados_copy['id_empleado'].isin([133,231]))
    )
]

##### Conclusión :
Se observa el ingreso de dos empleados con el mismo nombre, pero con distinto id_empleado y edad. Además, las fechas de ingreso no presentan coherencia entre sí en relación con las edades registradas. Por lo tanto, se interpreta que se trata de personas distintas.

##### empleado 'pedro martinez'

In [588]:
df_empleados_copy[df_empleados_copy['nombre']=='pedro martinez']

,id_empleado,nombre,edad,sexo,departamento,salario,fecha_ingreso
24,125,pedro martinez,30,m,recursos humanos,200000.0,2018-05-22
34,135,pedro martinez,40,m,it,130000.0,2020-08-05
53,154,pedro martinez,25,m,ventas,200000.0,2018-11-02
63,164,pedro martinez,28,f,recursos humanos,180000.0,2020-06-12
101,202,pedro martinez,40,f,it,200000.0,2018-01-13
103,204,pedro martinez,28,f,ventas,200000.0,2019-10-28
138,239,pedro martinez,25,f,it,120000.0,2019-04-06


In [590]:
df_empleados_copy = df_empleados_copy[df_empleados_copy['nombre'] != 'pedro martinez']

In [592]:
df_empleados_copy

,id_empleado,nombre,edad,sexo,departamento,salario,fecha_ingreso
2,103,juan perez,25,m,ventas,130000.0,2020-10-31
32,133,maria lopez,28,f,it,200000.0,2018-04-21
51,152,lucia fernandez,28,f,ventas,120000.0,2022-04-29
68,169,lucia fernandez,45,f,recursos humanos,180000.0,2020-04-11
84,185,ana gomez,25,f,it,200000.0,2018-01-13
113,214,lucia fernandez,45,f,ventas,120000.0,2019-08-22
126,227,ana gomez,30,f,recursos humanos,180000.0,2018-01-13
130,231,maria lopez,45,f,it,200000.0,2022-07-26
147,248,ana gomez,45,f,ventas,130000.0,2018-01-13


##### Conclusión :
Se detectaron múltiples registros asociados al nombre “Pedro Martínez” con inconsistencias significativas en edad, sexo, departamento, salario y fecha de ingreso. Dado que no fue posible reconstruir un perfil coherente ni identificar un patrón consistente entre los registros, se decidió eliminar dichos datos del dataset, considerando que no cumplen con los criterios mínimos de integridad y confiabilidad para el análisis.

#### Tareas realizadas :
En esta etapa se realizó una depuración de inconsistencias lógicas en los registros, tomando como referencia información proporcionada por la empresa. Para ello, se utilizaron dos tablas de validación: una correspondiente al género registrado por empleado y otra asociada a las escalas salariales definidas por departamento.

En los casos donde un mismo empleado presentaba inconsistencias en el género respecto al valor validado, dichos registros fueron eliminados por considerarse errores de carga. Asimismo, se contrastaron los salarios con los valores de referencia establecidos para cada área (Ventas, Recursos Humanos e IT), descartando aquellos registros que no respetaban la estructura salarial definida. Este procedimiento permitió reforzar la coherencia interna del dataset y garantizar mayor confiabilidad en el análisis posterior.

|masculino | femenino |  
| ---------| ------- |
| juan perez | lucia fernandez |
| pedro martinez | ana gomez |
| -- | maria lopez |


| Sueldo | departamento |
| ------ | -------- |
| 120.000 y 130.000 | ventas |
| 180.000 | recursos humanos |
| 200.000 | it |

#### Nueva columna 'antiguedad'

In [594]:
hoy = pd.Timestamp.today()

In [597]:
df_empleados_copy['antiguedad anios + meses']=(
        hoy - df_empleados_copy['fecha_ingreso']
    ).dt.days

In [599]:
df_empleados_copy['antiguedad en anios'] =(
    df_empleados_copy['antiguedad anios + meses'] / 365
    ).astype(int)

In [602]:
df_empleados_copy = df_empleados_copy.drop('antiguedad anios + meses',axis=1)

In [603]:
df_empleados_copy

,id_empleado,nombre,edad,sexo,departamento,salario,fecha_ingreso,antiguedad en anios
2,103,juan perez,25,m,ventas,130000.0,2020-10-31,5
32,133,maria lopez,28,f,it,200000.0,2018-04-21,7
51,152,lucia fernandez,28,f,ventas,120000.0,2022-04-29,3
68,169,lucia fernandez,45,f,recursos humanos,180000.0,2020-04-11,5
84,185,ana gomez,25,f,it,200000.0,2018-01-13,8
113,214,lucia fernandez,45,f,ventas,120000.0,2019-08-22,6
126,227,ana gomez,30,f,recursos humanos,180000.0,2018-01-13,8
130,231,maria lopez,45,f,it,200000.0,2022-07-26,3
147,248,ana gomez,45,f,ventas,130000.0,2018-01-13,8


#### Conclusión :
Se realiza una nueva tabla ,necesaria para nuestro analisis, la cual el valor sera de los años obtenidos referidos a la fecha de ingreso registrado

In [604]:
df_empleados_copy.to_csv('empleados_limpio.csv',index=False)